# Homework 7 -- ATOC 4815/5815

**Due: Thursday, 10/29, 11:59 pm.**

This homework is a partially-completed notebook, like HW6: some cells are already filled
in for you (things we covered in class, or that set up the problem), and some are marked
**`# TODO`** -- those are the parts you need to write.

**Skills practiced:**
- Extracting a 1-D transect ("flight line") of pixel values from a 2-D satellite image --
  the same nearest-grid-cell lookup from the gridding lectures, used in the other direction
- Computing and interpreting a power spectrum via the FFT (10/20 and 10/22 lecture)


## Context: what this builds on

On 10/6 and 10/13 we built the "find the nearest grid cell" lookup
(`grid_regular_annotated.py`) and used it to grid *scattered* MODIS data onto a regular
grid (`modis_terrain_corrected.py`, HW6 Task 1). On 10/15 and 10/22, in `HW6_seed.ipynb`,
we used that exact same lookup in the opposite direction: instead of gridding scattered
points onto an image's grid, we walked *along a straight line* through an image and
pulled out the pixel value under each point of that line -- the beginning of simulating
what a straight-and-level aircraft overflight would "see" looking straight down.

On 10/20 and 10/22 we covered Fourier series and transforms and power spectra, and used
`power_simple.py` (posted to Canvas/GitHub) to compute the power spectrum of a simple
test signal -- with the in-class task of refactoring that script into a reusable,
importable function.

**This homework combines both:** pull a straight-line transect through a real satellite
image of Hurricane Helene, then compute and interpret its power spectrum.

One new file comes with this notebook:
- **`power_spectrum_annotated.py`** -- the refactored, documented version of the
  `power_simple.py` function from class (same math, same result -- compare it against
  your own in-class version if you finished one). It's in the class GitHub repository's
  `code/` folder.


### Where to put this notebook

This notebook reads an image from a `dat/` folder and imports from
`power_spectrum_annotated.py`, so location matters, same as HW6:

1. `git pull` inside your cloned `ATOC-4815` repository to get this notebook (if posted
   there) and `power_spectrum_annotated.py`, then save/confirm `hw7.ipynb` is sitting
   **inside the `code/` folder** -- the same folder `weather.py` and
   `power_spectrum_annotated.py` are in.
2. Download **`helene.png`** from the HW7 page on Canvas -- like the MOD03 file for HW6,
   satellite imagery isn't part of the GitHub repo -- and put it in a folder named `dat/`
   that sits next to `py/` (one level up from this notebook).

```
ATOC-4815/
├── dat/
│   └── helene.png
└── py/
    ├── power_spectrum_annotated.py
    └── hw7.ipynb   <- this notebook
```


## Task 1 [30 pts] -- Pulling an aircraft overflight transect through Hurricane Helene

The image below (`helene.png`) is a satellite view of Hurricane Helene. `wesn`, `loc1`,
and `loc2` define a straight flight path from the storm's southwest quadrant to its
northeast quadrant -- roughly what a hurricane-hunter aircraft flying a straight overpass
leg might fly.

The cell below is **provided**: it loads the image and plots the flight line on top of
it, exactly like the setup in `HW6_seed.ipynb`.


In [ ]:
# --- Provided: load the image and show the flight line ---
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

png  = '../dat/helene.png'
wesn = [-89.0427, -80.9498, 22.4782, 30.9735]   # [west, east, south, north] of the image
loc1 = [-88.0, 22.5]    # flight track start point (lon, lat)
loc2 = [-82.1, 30.8]    # flight track end point (lon, lat)

img = mpimg.imread(png)
print('image shape:', img.shape)   # (rows, cols, channels) -- note PNGs often carry a 4th (alpha) channel

f0, ax0 = plt.subplots(1, 1, figsize=(6, 6))
ax0.imshow(img, extent=wesn)
ax0.plot([loc1[0], loc2[0]], [loc1[1], loc2[1]], 'r--')
ax0.set_title('Hurricane Helene -- planned overflight transect')
ax0.set_xlabel('Longitude'); ax0.set_ylabel('Latitude')
plt.show()


### Your turn

Using the same boundaries/midpoints lookup pattern as `grid_regular_annotated.py` and
`HW6_seed.ipynb`:

1. Build the image's pixel grid **boundaries** in lon/lat (`xg`, `yg`), the same way you
   did for the MODIS grid in HW6 (`nx`, `ny` from `img.shape`, then
   `np.linspace(wesn[...], ..., nx+1)`).
2. Sample `n` evenly-spaced (lon, lat) points between `loc1` and `loc2` with
   `np.linspace` -- pick `n` so that consecutive points are roughly 200 m apart (you'll
   need the approximate length of the flight line in km to figure out `n`; 1 degree of
   longitude or latitude here is roughly 111 km, same conversion as the distance formula
   you'll use in the next cell).
3. Loop over all `n` points and, for each one, find its nearest pixel index (`ix`, `iy`)
   in `xg`, `yg` -- same nearest-index lookup as before -- and record the pixel's **red,
   green, and blue** values (`img[..., 0]`, `img[..., 1]`, `img[..., 2]`) into three
   arrays `r`, `g`, `b`. Remember the row-order flip (`ny - 1 - iy`) from the gridding
   lectures: image rows count from the top, but `yg` was built south-to-north.


In [ ]:
# TODO: build the pixel grid boundaries (xg, yg), choose n, sample the flight line,
# and loop to fill r, g, b with the pixel values along it.

# TODO: your code here


Now convert your sampled points into a **physical along-track distance** in km (not
just lon/lat), and plot the three color channels against that distance -- the same
"overflight" panel style as the in-class gridding demo.

Hint: near latitude $\phi$, 1 degree of longitude is about $111 \cos(\phi)$ km, and 1
degree of latitude is about 111 km regardless of longitude. Use the straight-line
(Pythagorean) distance from the start point to build a cumulative distance array.


In [ ]:
# TODO: compute `dist` (same length as r/g/b), the along-track distance in km from the
# start of the flight line, and plot r, g, b vs. dist on one set of axes.

# TODO: your code here


## Task 2 [20 pts] -- Power spectrum of your transect

`power_spectrum_annotated.py` (in the `code/` folder, alongside this notebook) provides
`power_spectrum(T, data)` -- the function version of the `power_simple.py` demo from
class. Read its docstring and module comments; they walk through exactly how it connects
to the 10/20/10/22 lecture (Euler's formula, one-sided vs. two-sided spectra, the Nyquist
limit).


In [ ]:
# --- Provided: import the power spectrum function ---
from power_spectrum_annotated import power_spectrum


Using your **red-channel** transect `r` from Task 1:

1. Call `power_spectrum(T, r)`, where `T` is the *total* along-track distance in km
   (the last value of your `dist` array from Task 1).
2. Plot the result on **log-log** axes (`plt.loglog` or `plt.xscale('log')` +
   `plt.yscale('log')`) -- skip index 0 (the zero-frequency term; see the docstring for
   why). Label your axes, including units (your frequency axis is in cycles per km, i.e.
   1 / wavelength in km).
3. In a short markdown cell (or a comment), estimate roughly what wavelength (1 /
   frequency) carries the most power, and say whether that length scale matches
   something you can actually see in the image (the eye, a rainband spacing, the overall
   storm width, ...).


In [ ]:
# TODO: call power_spectrum on your red-channel transect, plot it on log-log axes
# with labeled units, and identify the dominant length scale.

# TODO: your code here


*Your answer here: dominant wavelength and what it corresponds to in the image.*


*Optional thought question (not graded, but worth thinking about): how would you expect
this power spectrum to change if your flight line passed straight through the eye of the
storm instead of grazing its edge? What about a flight line over open, storm-free ocean?*
